In [ ]:
# ------------------------------------------------------------
# Information Visualisation Coursework
# System C: Condition-Centric Exploration of Road Traffic Accidents
#
# Dataset:
# Leeds Road Traffic Accidents (Kaggle)
# https://www.kaggle.com/datasets/thedevastator/leeds-road-traffic-accidents-2009-2018
#
# Description:
# This notebook implements System C, a condition-driven interactive dashboard
# for exploring how road surface and lighting conditions relate to accident
# frequency, timing, severity, and location.
#
# Visualisation Tasks (Munzner, Ch.3 -- Actions + Targets):
#
#   T1 -- Explore spatial features  [Search -> Explore; Target: Feature]
#         Browse the accident map to discover geographic clusters and hotspot
#         areas where accidents concentrate across Leeds.
#         -> Map responds to condition and time selections; when nothing is
#            selected all accidents are visible for full spatial exploration.
#            Spatial brush on the map feeds back to the condition bar and
#            hour chart (bidirectional).
#
#   T2 -- Discover temporal distribution  [Analyse -> Discover; Target: Distribution]
#         Discover how accident frequency is distributed across hours of the day
#         to identify peak and low-risk time periods.
#         -> Hour bar chart filtered by the active condition selection.
#            Also responds to the map brush (bidirectional).
#
#   T3 -- Select a subset and summarise severity  [Search -> Browse; Query -> Summarise;
#         Target: Distribution]
#         Select a subset of accidents by road surface condition and/or hour
#         range and/or spatial region; summarise the severity distribution.
#         *** This task involves selecting a subset of the data. ***
#         -> Condition click (View 1) is the primary subset selection.
#            Hour brush (View 2) is secondary refinement.
#            Map brush (View 3) adds spatial refinement (bidirectional).
#
# Bidirectional Linking:
#   surface_sel (condition point) -- condition bar produces, hour + map + severity consume
#   time_brush  (hour interval)   -- hour chart produces, map + severity consume
#   map_brush   (spatial interval) -- map produces, condition bar + hour chart consume
#
#   Bidirectional pairs:
#     Condition bar <-> Map: condition click filters map; map brush reshapes
#                            the condition bar to show conditions in that region.
#     Hour chart    <-> Map: time brush filters map; map brush reshapes
#                            the hour distribution for that region.
#
# Design Decisions (distinguishing C from A and B):
#   D1. Primary view:       A = spatial scatter map
#                           B = time-of-day period bar + hour bar
#                           C = road surface condition stacked bar
#
#   D2. Selection mechanism: A = spatial interval brush + temporal interval brush
#                            B = time-period point click + temporal interval brush
#                                + spatial interval brush
#                            C = categorical point click + temporal interval brush
#                                + spatial interval brush
#
#   D3. Colour in primary:  A = severity (Slight/Serious/Fatal) on the map
#                           B = time-of-day period category on the period bar
#                           C = lighting condition (Daylight/Darkness) on condition bar
#
#   D4. Role of the map:    A = PRIMARY selection surface -- map is the main
#                               interaction point
#                           B = BIDIRECTIONAL node -- responds to time/period
#                               and feeds back to hour chart
#                           C = BIDIRECTIONAL hub -- responds to condition + time,
#                               AND its spatial brush updates BOTH the condition
#                               bar (reshowing conditions in region) AND the hour
#                               chart (reshowing hours for region)
#
#   D5. Severity chart:     A = filtered by spatial region AND hour range
#                           B = filtered by period AND time window AND spatial region
#                           C = filtered by road surface condition AND time window
#                               AND spatial region
#
#   D6. Linking direction:  A = spatial <-> temporal (bidirectional pair)
#                           B = hierarchical cascade: period -> time -> space,
#                               plus space -> time
#                           C = dual bidirectional: condition <-> space AND
#                               time <-> space (map is the shared bidirectional hub)
#
# Libraries used:
# pandas -- data processing
# altair -- interactive visualisation
# ------------------------------------------------------------

In [ ]:
# !pip install altair pandas

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

df = pd.read_csv("data/data.csv")
print(df.shape)
df.head()

In [ ]:
# Keep only rows with valid coordinates, hour, severity, and road surface
df = df[df["has_coords"] == True].copy()
df = df.dropna(subset=["hour", "severity", "road_surface", "lighting_conditions"])
df["hour"] = df["hour"].astype(int)

# Simplify lighting to two categories
df["lighting"] = df["lighting_conditions"].apply(
    lambda x: "Daylight" if "Daylight" in str(x) else "Darkness"
)

# Canonical road-surface order (dry -> most hazardous)
SURFACE_ORDER = ["Dry", "Wet / Damp", "Frost / Ice", "Snow"]

print("Rows after filtering:", len(df))
print("Road surface values:", df["road_surface"].unique().tolist())
print("Lighting values:    ", df["lighting"].unique().tolist())

In [ ]:
# ---- Selections (three independent, composing selections) ---------------

# T3 primary: condition click -- road surface category
# empty=True: all data visible when nothing clicked
surface_sel = alt.selection_point(
    name="surface_sel",
    fields=["road_surface"],
    empty=True
)

# T3 secondary: hour interval brush
time_brush = alt.selection_interval(
    name="time_brush",
    encodings=["x"]
)

# T3 spatial + bidirectional: map brush feeds back to condition bar + hour chart
map_brush = alt.selection_interval(name="map_brush")

In [ ]:
# ---- View 1: Condition Bar -- PRIMARY (T3 subset selection) --------------
# Produces: surface_sel -> filters hour chart, map, severity
# Consumes: map_brush   -> reshows condition breakdown for the brushed region
#
# BIDIRECTIONAL with map:
#   Forward:  click a surface bar -> map shows only that surface's accidents
#   Backward: brush a map region  -> condition bar reshapes to show what
#             surfaces are present in that geographic area
#
# D3: Colour encodes lighting condition (Daylight vs Darkness) -- not severity.

condition_bar = (
    alt.Chart(df,
        title="T3 / T1 -- Road Surface & Lighting (click surface to select; map brush reshapes)")
    .mark_bar()
    .transform_filter(map_brush)          # BIDIRECTIONAL: respond to spatial brush
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["road_surface", "lighting"]
    )
    .encode(
        x=alt.X("road_surface:N", sort=SURFACE_ORDER, title="Road Surface Condition"),
        y=alt.Y("accident:Q", title="Number of Accidents", stack="zero"),
        color=alt.Color(
            "lighting:N",
            title="Lighting",
            scale=alt.Scale(
                domain=["Daylight", "Darkness"],
                range=["#F5CBA7", "#1A252F"]
            )
        ),
        opacity=alt.condition(surface_sel, alt.value(1.0), alt.value(0.25)),
        tooltip=[
            alt.Tooltip("road_surface:N", title="Road Surface"),
            alt.Tooltip("lighting:N",     title="Lighting"),
            alt.Tooltip("accident:Q",     title="Accidents"),
        ]
    )
    .add_params(surface_sel)             # produces condition selection
    .properties(width=320, height=220)
)

In [ ]:
# ---- View 2: Stacked Area -- T2 (bidirectional, filtered by condition + map) --
# Consumes: surface_sel (condition) + map_brush (spatial -- bidirectional)
# Produces: time_brush  (temporal selection -> filters map + severity)
#
# Stacked area shows how each road surface condition contributes to accident
# frequency across the day -- reveals when hazardous conditions (frost, snow)
# peak relative to dry/wet roads.
#
# BIDIRECTIONAL with map:
#   Forward:  drag hour brush -> map shows only accidents in that time range
#   Backward: brush map region -> area chart reshapes to show temporal
#             distribution of accidents within that geographic region
#
# Opacity dims the area outside the time_brush selection.

hour_chart = (
    alt.Chart(df,
        title="T2 — Accident Profile by Hour & Condition (drag to select; map brush reshapes)")
    .mark_area()
    .transform_filter(surface_sel)
    .transform_filter(map_brush)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["hour", "road_surface"]
    )
    .encode(
        x=alt.X("hour:Q", title="Hour of Day (0-23)", scale=alt.Scale(domain=[0, 23])),
        y=alt.Y("accident:Q", title="Number of Accidents", stack="zero"),
        color=alt.Color(
            "road_surface:N",
            sort=SURFACE_ORDER,
            scale=alt.Scale(
                domain=SURFACE_ORDER,
                range=["#E59866", "#5B9BD5", "#85C1E9", "#AED6F1"]
            ),
            title="Road Surface"
        ),
        opacity=alt.condition(time_brush, alt.value(0.85), alt.value(0.2)),
        tooltip=[
            alt.Tooltip("hour:Q",         title="Hour"),
            alt.Tooltip("road_surface:N", title="Road Surface"),
            alt.Tooltip("accident:Q",     title="Accidents")
        ]
    )
    .add_params(time_brush)
    .properties(width=420, height=220)
)

In [ ]:
# ---- View 3: Spatial Map -- T1 (BIDIRECTIONAL hub) -----------------------
# Consumes: surface_sel + time_brush
# Produces: map_brush -> feeds back to BOTH condition bar (View 1) AND
#                        hour chart (View 2)
#
# D4: Map is the BIDIRECTIONAL hub in System C -- it has two inputs
#     (condition and time) and its output (map_brush) updates two other views.
# D6: Dual bidirectional: condition <-> space AND time <-> space.

xmin, xmax = int(df["easting"].min()),  int(df["easting"].max())
ymin, ymax = int(df["northing"].min()), int(df["northing"].max())

map_chart = (
    alt.Chart(df,
        title="T1 -- Accident Locations (drag map to reshape condition bar & hour chart)")
    .mark_circle(size=14, opacity=0.45)
    .transform_filter(surface_sel)
    .transform_filter(time_brush)
    .encode(
        x=alt.X("easting:Q",  title="Grid Ref: Easting",
                scale=alt.Scale(domain=[xmin, xmax])),
        y=alt.Y("northing:Q", title="Grid Ref: Northing",
                scale=alt.Scale(domain=[ymin, ymax])),
        color=alt.Color(
            "severity:N",
            title="Severity",
            scale=alt.Scale(
                domain=["Slight", "Serious", "Fatal"],
                range=["#4CAF50", "#FC8D59", "#D73027"]
            )
        ),
        tooltip=[
            alt.Tooltip("accident_date:T",  title="Date",        format="%Y-%m-%d"),
            alt.Tooltip("hour:O",           title="Hour"),
            alt.Tooltip("severity:N",       title="Severity"),
            alt.Tooltip("road_surface:N",   title="Road Surface"),
            alt.Tooltip("lighting:N",       title="Lighting"),
            alt.Tooltip("num_casualties:Q", title="Casualties"),
        ]
    )
    .add_params(map_brush)               # produces spatial selection (bidirectional)
    .properties(width=470, height=320)
)

In [ ]:
# ---- View 4: Severity Breakdown -- T3 (filtered by all three selections) --
# Consumes: surface_sel + time_brush + map_brush
# D5: Filtered by road surface condition AND time window AND spatial region
#     -- unique to System C (three composing filters).

severity_chart = (
    alt.Chart(df, title="T3 -- Severity of Selected Subset")
    .mark_bar()
    .transform_filter(surface_sel)
    .transform_filter(time_brush)
    .transform_filter(map_brush)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["severity"]
    )
    .encode(
        x=alt.X("severity:N", sort=["Fatal", "Serious", "Slight"], title="Severity"),
        y=alt.Y("accident:Q", title="Number of Accidents"),
        color=alt.Color(
            "severity:N",
            scale=alt.Scale(
                domain=["Slight", "Serious", "Fatal"],
                range=["#4CAF50", "#FC8D59", "#D73027"]
            ),
            legend=None
        ),
        tooltip=[
            alt.Tooltip("severity:N", title="Severity"),
            alt.Tooltip("accident:Q", title="Accidents"),
        ]
    )
    .properties(width=270, height=320)
)

In [ ]:
# ---- Dashboard Layout ---------------------------------------------------
#
# +-----------------------------+------------------------------+
# | View 1: Condition Bar       | View 2: Hour Bar             |
# | T3 click / map feeds back   | T2 brush / map feeds back    |
# +-----------------------------+------------------+-----------+
# | View 3: Spatial Map (T1)                       | View 4:   |
# | Bidirectional hub -- drag to update V1 + V2    | Severity  |
# +------------------------------------------------+-----------+
#
# D4: 2x2 grid layout. Map is bottom-left (bidirectional hub).
#     A uses map as top (primary source). B uses period bar as top.

top_row    = alt.hconcat(condition_bar, hour_chart).resolve_scale(color="independent")
bottom_row = alt.hconcat(map_chart, severity_chart).resolve_scale(color="independent")

system_c = (
    alt.vconcat(top_row, bottom_row)
    .configure_title(fontSize=12, anchor="start")
    .configure_axis(gridOpacity=0.25)
    .configure_view(stroke=None)
)

system_c